# YOLO26 + EfficientViT

执行顺序：初始化拉取代码 → 基线检查 → 安装 timm → 创建适配器 → 测试适配器 → 注册模块并修改 tasks.py → 创建 YAML → 构建和前向验证。

适配方案：EfficientViT 输出 `[P3, P4, P5]` 列表，使用 Ultralytics 已有的 `Index` 模块取出三层特征；不替换整个 `tasks.py`，也不强制修改 `_predict_once()`。


In [1]:
# 新 Colab 运行时初始化：拉取 YOLO26 和 EfficientViT
from pathlib import Path
import subprocess

CONTENT = Path("/content")
ULTRALYTICS = CONTENT / "ultralytics"
EFFICIENTVIT = CONTENT / "efficientvit"

def run(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, cwd=cwd, shell=True, check=True)

if not (ULTRALYTICS / ".git").exists():
    run("git clone --depth 1 https://github.com/ultralytics/ultralytics.git /content/ultralytics")
else:
    print("已有 /content/ultralytics，跳过 clone")

if not (EFFICIENTVIT / ".git").exists():
    run("git clone --depth 1 https://github.com/mit-han-lab/efficientvit.git /content/efficientvit")
else:
    print("已有 /content/efficientvit，跳过 clone")

run("git fetch --depth 1 origin main", cwd=ULTRALYTICS)
run("git checkout --detach origin/main", cwd=ULTRALYTICS)
assert (ULTRALYTICS / "ultralytics" / "cfg" / "models" / "26" / "yolo26.yaml").exists()
run("python -m pip install -q -e /content/ultralytics --no-deps")
print("YOLO26:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ULTRALYTICS, text=True).strip())
print("EfficientViT:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=EFFICIENTVIT, text=True).strip())


已有 /content/ultralytics，跳过 clone
已有 /content/efficientvit，跳过 clone
$ git fetch --depth 1 origin main
$ git checkout --detach origin/main
$ python -m pip install -q -e /content/ultralytics --no-deps
YOLO26: 85c3d80d7106706112cb1bf67ce090939dc38b20
EfficientViT: de7d7733cc0329f391b33f1f459271562ec27bd5


In [2]:
%cd /content/ultralytics

import sys
from pathlib import Path

repo = "/content/ultralytics"
if repo not in sys.path:
    sys.path.insert(0, repo)

import numpy as np
import torch
import ultralytics
from ultralytics import YOLO

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("ultralytics:", ultralytics.__file__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

model = YOLO("yolo26n.pt")
print("baseline model loaded")


/content/ultralytics
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 2.1.3
ultralytics: /content/ultralytics/ultralytics/__init__.py
torch: 2.11.0+cu130
cuda: True
gpu: Tesla T4
baseline model loaded


In [3]:
%pip install -q timm

import timm
names = [name for name in timm.list_models() if "efficientvit_m" in name.lower()]
print("available models:", names)
assert "efficientvit_m0" in names, "efficientvit_m0 is unavailable in this timm installation"


available models: ['efficientvit_m0', 'efficientvit_m1', 'efficientvit_m2', 'efficientvit_m3', 'efficientvit_m4', 'efficientvit_m5']


In [4]:
from pathlib import Path

root = Path("/content/ultralytics")
extra = root / "ultralytics" / "nn" / "modules" / "extramodules"
extra.mkdir(parents=True, exist_ok=True)
(extra / "__init__.py").touch()
print(extra)


/content/ultralytics/ultralytics/nn/modules/extramodules


In [5]:
from pathlib import Path

root = Path("/content/ultralytics")
adapter = r'''# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
from __future__ import annotations

import torch
from torch import nn
import torch.nn.functional as F
import timm

__all__ = [
    "EfficientViT_M0", "EfficientViT_M1", "EfficientViT_M2",
    "EfficientViT_M3", "EfficientViT_M4", "EfficientViT_M5",
]


class _EfficientViTBackbone(nn.Module):
    def __init__(self, variant: str, pretrained: bool = False):
        super().__init__()
        self.model = timm.create_model(
            variant,
            pretrained=pretrained,
            img_size=640,
            features_only=True,
            out_indices=(0, 1, 2),
        )
        reductions = list(self.model.feature_info.reduction())
        channels = list(self.model.feature_info.channels())
        if reductions != [16, 32, 64]:
            raise RuntimeError(f"{variant} unexpected reductions: {reductions}")
        self.channel = [channels[0], channels[0], channels[1]]

    def forward(self, x: torch.Tensor):
        # timm EfficientViT-M uses fixed window sizes configured for 640x640.
        # YOLO probes the model with smaller tensors while calculating strides,
        # so resize internally and restore P3/P4/P5 to the input resolution.
        h, w = x.shape[-2:]
        x640 = F.interpolate(x, size=(640, 640), mode="bilinear", align_corners=False)
        features = self.model(x640)
        p4 = features[0]
        p5 = features[1]
        p3 = F.interpolate(p4, scale_factor=2, mode="nearest")
        p3 = F.interpolate(p3, size=(max(h // 8, 1), max(w // 8, 1)), mode="nearest")
        p4 = F.interpolate(p4, size=(max(h // 16, 1), max(w // 16, 1)), mode="nearest")
        p5 = F.interpolate(p5, size=(max(h // 32, 1), max(w // 32, 1)), mode="nearest")
        return [p3, p4, p5]


class EfficientViT_M0(_EfficientViTBackbone):
    def __init__(self):
        super().__init__("efficientvit_m0", pretrained=False)


class EfficientViT_M1(_EfficientViTBackbone):
    def __init__(self):
        super().__init__("efficientvit_m1", pretrained=False)


class EfficientViT_M2(_EfficientViTBackbone):
    def __init__(self):
        super().__init__("efficientvit_m2", pretrained=False)


class EfficientViT_M3(_EfficientViTBackbone):
    def __init__(self):
        super().__init__("efficientvit_m3", pretrained=False)


class EfficientViT_M4(_EfficientViTBackbone):
    def __init__(self):
        super().__init__("efficientvit_m4", pretrained=False)


class EfficientViT_M5(_EfficientViTBackbone):
    def __init__(self):
        super().__init__("efficientvit_m5", pretrained=False)
'''
path = root / "ultralytics" / "nn" / "modules" / "extramodules" / "efficientvit.py"
path.write_text(adapter, encoding="utf-8")
print(path)


/content/ultralytics/ultralytics/nn/modules/extramodules/efficientvit.py


In [6]:
import importlib.util
import torch

adapter_path = "/content/ultralytics/ultralytics/nn/modules/extramodules/efficientvit.py"
spec = importlib.util.spec_from_file_location("efficientvit_adapter", adapter_path)
efficientvit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(efficientvit)

assert issubclass(efficientvit.EfficientViT_M0, torch.nn.Module)
backbone = efficientvit.EfficientViT_M0().cuda().eval()
x = torch.randn(1, 3, 640, 640, device="cuda")
with torch.inference_mode():
    outputs = backbone(x)

print("channels:", backbone.channel)
for i, output in enumerate(outputs):
    print(i, tuple(output.shape))
assert [tuple(output.shape[-2:]) for output in outputs] == [(80, 80), (40, 40), (20, 20)]
print("adapter test passed")


channels: [64, 64, 128]
0 (1, 64, 80, 80)
1 (1, 64, 40, 40)
2 (1, 128, 20, 20)
adapter test passed


## 修改 Ultralytics 模块

下面的单元会备份并修改 `__init__.py`、`tasks.py`，然后创建 YAML。执行后重启运行时，再执行最后的模型构建单元。


In [7]:
from pathlib import Path
import shutil

root = Path("/content/ultralytics")
modules_init = root / "ultralytics" / "nn" / "modules" / "__init__.py"
tasks = root / "ultralytics" / "nn" / "tasks.py"

registration = '''
from .extramodules.efficientvit import (
    EfficientViT_M0,
    EfficientViT_M1,
    EfficientViT_M2,
    EfficientViT_M3,
    EfficientViT_M4,
    EfficientViT_M5,
)
'''

if "from .extramodules.efficientvit import" not in modules_init.read_text(encoding="utf-8"):
    shutil.copy2(modules_init, str(modules_init) + ".bak")
    with modules_init.open("a", encoding="utf-8") as f:
        f.write(registration)

text = tasks.read_text(encoding="utf-8")
if "efficientvit_modules = frozenset" not in text:
    shutil.copy2(tasks, str(tasks) + ".bak")
    import_marker = "from ultralytics.nn.autobackend import check_class_names"
    imports = '''from ultralytics.nn.modules.extramodules.efficientvit import (
    EfficientViT_M0,
    EfficientViT_M1,
    EfficientViT_M2,
    EfficientViT_M3,
    EfficientViT_M4,
    EfficientViT_M5,
)
'''
    text = text.replace(import_marker, imports + "\n" + import_marker, 1)
    loop_marker = "    for i, (f, n, m, args) in enumerate(d[\"backbone\"] + d[\"head\"]):  # from, number, module, args"
    collection = '''    efficientvit_modules = frozenset({
        EfficientViT_M0, EfficientViT_M1, EfficientViT_M2,
        EfficientViT_M3, EfficientViT_M4, EfficientViT_M5,
    })
'''
    if loop_marker not in text:
        raise RuntimeError("parse_model loop marker not found")
    text = text.replace(loop_marker, collection + loop_marker, 1)
    n_marker = "        n = n_ = max(round(n * depth), 1) if n > 1 else n  # depth gain"
    text = text.replace(n_marker, n_marker + "\n        m_ = None", 1)
    branch_marker = "        if m in base_modules:"
    branch = '''        if m in efficientvit_modules:
            if n != 1:
                raise ValueError("EfficientViT backbone only supports repeats=1")
            m_ = m(*args)
            c2 = m_.channel
        elif m in base_modules:
'''
    if branch_marker not in text:
        raise RuntimeError("base_modules branch not found")
    text = text.replace(branch_marker, branch, 1)
    instantiate = "        m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module"
    replacement = "        if m_ is None:\n            m_ = torch.nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)  # module"
    if instantiate not in text:
        raise RuntimeError("module instantiation line not found")
    text = text.replace(instantiate, replacement, 1)
    tasks.write_text(text, encoding="utf-8")

print("registered and patched; restart the runtime before importing ultralytics again")


registered and patched; restart the runtime before importing ultralytics again


In [8]:
from pathlib import Path

cfg = Path("/content/ultralytics/ultralytics/cfg/models/26/yolo26_efficientvit.yaml")
cfg.write_text("""nc: 80
end2end: True
reg_max: 1

backbone:
  - [-1, 1, EfficientViT_M0, []]

head:
  - [0, 1, Index, [64, 0]]
  - [0, 1, Index, [64, 1]]
  - [0, 1, Index, [128, 2]]
  - [[1, 2, 3], 1, Detect, [nc]]
""", encoding="utf-8")
print(cfg)


/content/ultralytics/ultralytics/cfg/models/26/yolo26_efficientvit.yaml


## 重启后构建和验证模型

执行“重启运行时”，然后从初始化单元开始执行；适配器文件、注册、`tasks.py` 补丁和 YAML 会保留在 `/content` 中，但 Python 导入缓存会被清除。


In [9]:
%cd /content/ultralytics
from ultralytics import YOLO

model = YOLO("/content/ultralytics/ultralytics/cfg/models/26/yolo26_efficientvit.yaml")
model.info(verbose=True)
print("model build passed")


/content/ultralytics
YOLO26_efficientvit summary: 329 layers, 2,999,832 parameters, 2,999,832 gradients
model build passed


In [10]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
dummy = torch.randn(1, 3, 640, 640, device=device)
with torch.inference_mode():
    result = model(dummy)
print("forward passed")



WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.777966022491455. Dividing input by 255.
0: 640x640 (no detections), 26.8ms
Speed: 0.1ms preprocess, 26.8ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)
forward passed


In [ ]:
#归一化前向输入测试
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.rand(1, 3, 640, 640, device=device)

with torch.inference_mode():
    result = model(x)

print("normalized forward passed")


0: 640x640 (no detections), 26.7ms
Speed: 0.1ms preprocess, 26.7ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)
normalized forward passed


In [12]:
#反向传播测试
model.train()
x = torch.rand(2, 3, 640, 640, device="cuda")

outputs = model.model(x)
loss = sum(
    output.float().mean()
    for output in outputs
    if torch.is_tensor(output)
)

loss.backward()

print("backward passed")

Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/ultralytics/ultralytics/cfg/models/26/yolo26_efficientvit.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, n

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [13]:
#冒烟测试
from ultralytics import YOLO

model = YOLO(
    "/content/ultralytics/ultralytics/cfg/models/26/yolo26_efficientvit.yaml"
)

model.train(
    data="coco8.yaml",
    epochs=1,
    imgsz=640,
    batch=2,
    workers=2,
    device=0,
    project="/content/runs",
    name="yolo26_efficientvit_smoke",
)

Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/ultralytics/ultralytics/cfg/models/26/yolo26_efficientvit.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26_efficientvit

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0, 16, 17, 20, 25, 58])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a248af448a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [14]:
from pathlib import Path
from ultralytics import YOLO

best = Path("/content/runs/yolo26_efficientvit_smoke/weights/best.pt")
print(best.exists(), best)

if best.exists():
    reloaded = YOLO(str(best))
    reloaded.info(verbose=True)

True /content/runs/yolo26_efficientvit_smoke/weights/best.pt
YOLO26_efficientvit summary: 329 layers, 2,999,832 parameters, 0 gradients
